# Kardynalność cech – zbiór Titanic

Ćwiczenie dotyczy kardynalności (liczności) cech, czyli liczby różnych wartości, które zmienna przyjmuje w zbiorze danych. Analizę wykonano na zbiorze Titanic, który zawiera dane o 1309 pasażerach.

## Przygotowanie danych

In [1]:
import pandas as pd
import numpy as np
import arff

In [2]:
filepath = "data/Titanic.arff"
with open(filepath, "r", encoding="utf-8") as f:
    dataset = arff.load(f)

df = pd.DataFrame(dataset["data"], columns=[col[0] for col in dataset["attributes"]])

## 1. Liczebność etykiet poszczególnych zmiennych

In [3]:
for col in df:
    print('Liczba etykiet zmiennej {}:{}'.format(col,len(df[col].unique())))

Liczba etykiet zmiennej pclass:3
Liczba etykiet zmiennej survived:2
Liczba etykiet zmiennej name:1307
Liczba etykiet zmiennej sex:2
Liczba etykiet zmiennej age:99
Liczba etykiet zmiennej sibsp:7
Liczba etykiet zmiennej parch:8
Liczba etykiet zmiennej ticket:929
Liczba etykiet zmiennej fare:282
Liczba etykiet zmiennej cabin:187
Liczba etykiet zmiennej embarked:4
Liczba etykiet zmiennej boat:28
Liczba etykiet zmiennej body:122
Liczba etykiet zmiennej home.dest:370


## 2. Liczba wszystkich pasażerów

In [4]:
print('Liczba wszystkich pasażerów: {}'.format(len(df)))

Liczba wszystkich pasażerów: 1309


### Uwaga: powtarzające się imiona i nazwiska

Liczba unikalnych imion i nazwisk (1307) jest o dwa mniejsza niż liczba wszystkich pasażerów (1309), co może sugerować zduplikowane rekordy.

In [5]:
df["name"].value_counts()

name
Connolly, Miss. Kate              2
Kelly, Mr. James                  2
Allen, Miss. Elisabeth Walton     1
Allison, Master. Hudson Trevor    1
Allison, Miss. Helen Loraine      1
                                 ..
Zabour, Miss. Hileni              1
Zabour, Miss. Thamine             1
Zakarian, Mr. Mapriededer         1
Zakarian, Mr. Ortin               1
Zimmerman, Mr. Leo                1
Name: count, Length: 1307, dtype: int64

In [6]:
df[df["name"].isin(["Connolly, Miss. Kate","Kelly, Mr. James"])]

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
725,3.0,1,"Connolly, Miss. Kate",female,22.0,0.0,0.0,370373,7.7500,NaN,Q,13,NaN,Ireland
726,3.0,0,"Connolly, Miss. Kate",female,30.0,0.0,0.0,330972,7.6292,NaN,Q,NaN,NaN,Ireland
924,3.0,0,"Kelly, Mr. James",male,34.5,0.0,0.0,330911,7.8292,NaN,Q,NaN,70.0,NaN
925,3.0,0,"Kelly, Mr. James",male,44.0,0.0,0.0,363592,8.0500,NaN,S,NaN,NaN,NaN


Powtarzają się dwie pozycje: *Connolly, Miss. Kate* oraz *Kelly, Mr. James*. Pozostałe kolumny (wiek, numer biletu, port zaokrętowania) pokazują jednak, że w obu przypadkach chodzi o różne osoby o tych samych danych osobowych, a nie o zduplikowane rekordy. Nie ma więc powodu, aby usuwać te wiersze.

In [7]:
print(df["cabin"].value_counts())
print(f"\n{(np.mean(df["cabin"].value_counts() == 1) *100):.2f}% numerów kabin występuje tylko raz w danych, więc ciężko oszacować ich wpływ na przeżycie")

cabin
C23 C25 C27        6
B57 B59 B63 B66    5
G6                 5
C22 C26            4
B96 B98            4
                  ..
F E57              1
F E46              1
F E69              1
E10                1
F38                1
Name: count, Length: 186, dtype: int64

57.53% numerów kabin występuje tylko raz w danych, więc ciężko oszacować ich wpływ na przeżycie


## 3. Komentarz do wyników z punktów 1 i 2

W zbiorze znajduje się **1309 pasażerów**.

Metoda `unique()` traktuje brakujące wartości (`NaN`) jako osobną etykietę. W związku z tym zmienne zawierające braki danych (np. `embarked`, `cabin`) mają w wyniku o jedną etykietę więcej niż liczba faktycznie występujących kategorii.

### Zmienne o małej kardynalności

Zmienne te przyjmują od kilku do kilkunastu różnych wartości, niezależnie od liczby pasażerów.

* `survived` (2) – informacja o przeżyciu pasażera (0 lub 1),
* `sex` (2) – płeć pasażera,
* `pclass` (3) – klasa podróży (1., 2. lub 3.),
* `embarked` (4) – port zaokrętowania (trzy porty oraz brak danych),
* `sibsp` (7) – liczba rodzeństwa lub małżonków na pokładzie,
* `parch` (8) – liczba rodziców lub dzieci na pokładzie,
* `boat` (28) – numer szalupy ratunkowej. Liczba etykiet jest większa niż w przypadku pozostałych zmiennych z tej grupy, jednak nadal niewielka w porównaniu z liczbą rekordów.

### Zmienne o dużej kardynalności

Zmienne te przyjmują od kilkuset do ponad tysiąca różnych wartości. Wiele z nich występuje tylko raz lub kilka razy.

* `name` (1307) – niemal każdy pasażer ma unikalne imię i nazwisko. Różnica względem liczby rekordów (1309) wynika z dwóch powtarzających się wartości, omówionych wcześniej,
* `ticket` (929) – numer biletu. Nie każdy pasażer podróżował samodzielnie – część osób podróżowała na wspólnych biletach rodzinnych lub grupowych,
* `home.dest` (370) – miejsce zamieszkania lub cel podróży,
* `cabin` (187) – numer kabiny. Duża liczba etykiet wynika z liczby kabin na statku. Podobnie jak w przypadku biletów, zdarzało się, że kilka osób (np. członkowie rodziny) było przypisanych do tej samej kabiny,
* `body` (122) – numer identyfikacyjny odnalezionego ciała.

### Zmienne liczbowe ciągłe

Zmienne `age` (99) i `fare` (282) formalnie należą do grupy o dużej kardynalności. Wynika to jednak z ciągłego charakteru reprezentowanych przez nie wielkości (wieku i ceny biletu), a nie z dużej liczby kategorii. Z tego względu nie traktuję ich jako zmiennych jakościowych.

## 4. Unikalne etykiety zmiennej `cabin`

In [8]:
# unique() zwraca unikalne wartości; np.asarray zapewnia wynik w postaci tablicy NumPy
cabin_labels = np.asarray(df["cabin"].unique(), dtype=object)

print(type(cabin_labels))
print('Liczba etykiet zmiennej cabin: {}'.format(len(cabin_labels)))
print('Pierwsze 20 etykiet:')
print(cabin_labels[:20])

<class 'numpy.ndarray'>
Liczba etykiet zmiennej cabin: 187
Pierwsze 20 etykiet:
['B5' 'C22 C26' 'E12' 'D7' 'A36' 'C101' nan 'C62 C64' 'B35' 'A23'
 'B58 B60' 'D15' 'C6' 'D35' 'C148' 'C97' 'B49' 'C99' 'C52' 'T']


Zmienna `cabin` ma 187 unikalnych etykiet. Jedna z nich to `NaN` (brak danych), więc faktycznych oznaczeń kabin jest 186.

Metoda `unique()` zwraca wartości w kolejności ich pierwszego wystąpienia w zbiorze. W nowszych wersjach pandas dla kolumn tekstowych zwraca ona tablicę typu pandasowego, dlatego wynik jawnie zamieniam na tablicę NumPy (`np.asarray`).

## 5. Redukcja liczby etykiet – zmienna `CabinReduced`

Oznaczenie kabiny ma format *litera + numer* (np. `C22`), przy czym litera wskazuje pokład. Nowa zmienna `CabinReduced` zawiera wyłącznie pierwszą literę oznaczenia kabiny, a dla braków danych pozostawia `NaN`.

In [9]:
df["CabinReduced"] = df["cabin"].astype(str).str[0]

In [10]:
df[["cabin", "CabinReduced"]].head(20)

,cabin,CabinReduced
0,B5,B
1,C22 C26,C
2,C22 C26,C
3,C22 C26,C
4,C22 C26,C
5,E12,E
6,D7,D
7,A36,A
8,C101,C
9,NaN,NaN


Część wpisów w kolumnie `cabin` zawiera po kilka numerów kabin – warto sprawdzić, czy występują wpisy przypisane do różnych pokładów.

In [11]:
cabins = df["cabin"].dropna()
n_cabins = cabins.str.split().map(len)
decks = cabins.str.split().map(lambda parts: sorted({p[0] for p in parts}))
mixed = cabins[decks.map(len) > 1]

print('Wpisy z kilkoma kabinami: {} z {}'.format((n_cabins > 1).sum(), len(cabins)))
print('Wpisy z kilkoma pokładami: {} z {}'.format(len(mixed), len(cabins)))
print('Przypisane kabiny o różnych oznaczeniach liter: ' + ', '.join(mixed))

Wpisy z kilkoma kabinami: 41 z 295
Wpisy z kilkoma pokładami: 7 z 295
Przypisane kabiny o różnych oznaczeniach liter: F G63, F G63, F E57, F E46, F G73, F E69, F G73


## 6. Zmiana kardynalności zmiennej opisującej kabiny

In [12]:
for col in ["cabin", "CabinReduced"]:
    print('Liczba etykiet zmiennej {}: {}'.format(col, len(df[col].unique())))

n_cabin = len(df["cabin"].unique())
n_reduced = len(df["CabinReduced"].unique())
reduction = (n_cabin - n_reduced) / n_cabin * 100
print('Kardynalność zmiennej cabin zmniejszono o {:.2f}%.'.format(reduction))
labels = ', '.join(map(str, df["CabinReduced"].unique()))
print(f'Pozostałe etykiety: {labels}')

Liczba etykiet zmiennej cabin: 187
Liczba etykiet zmiennej CabinReduced: 9
Kardynalność zmiennej cabin zmniejszono o 95.19%.
Pozostałe etykiety: B, C, E, D, A, nan, T, F, G


Liczba etykiet spadła ze 187 do 9, czyli o 95,19%. Po redukcji pozostało osiem pokładów (A–G oraz T) i etykieta oznaczająca brak danych.

## 7. Uzasadnienie redukcji oraz jej skutki

### Dlaczego redukuję zmienną `cabin`

Zmienna `cabin` ma dużą kardynalność (187 etykiet), ale w odróżnieniu od większości innych takich zmiennych po uproszczeniu zachowuje informację, która może mieć znaczenie w analizie: oznaczenie pokładu, na którym znajdowała się kabina pasażera. Pierwsza litera numeru kabiny pozwala zmniejszyć liczbę etykiet o 95,19% (do 9) i jednocześnie zachować tę informację.

Pozostałe zmienne o dużej kardynalności nie nadają się do takiej redukcji:

* `name` i `ticket` są w praktyce identyfikatorami pasażerów, więc w surowej postaci nie wnoszą informacji przydatnej w modelowaniu (z `name` można co najwyżej wyodrębnić tytuł, np. *Mr.* lub *Miss.*),
* `body` i `boat` zawierają informacje znane dopiero po katastrofie: numer ciała mają tylko osoby, które zginęły, a numer szalupy – niemal wyłącznie osoby, które się uratowały. Użycie tych zmiennych do przewidywania przeżycia spowodowałoby wyciek danych (*data leakage*),
* `home.dest` jest silnie rozdrobniona (370 etykiet) i nie ma w niej prostego, naturalnego sposobu redukcji takiego jak pierwsza litera w numerze kabiny,
* `age` i `fare` są zmiennymi ciągłymi, więc zamiast redukcji etykiet stosuje się dla nich podział na przedziały (np. bilety tanie, średnie i drogie).

### Wpływ na przyszłe analizy

* **Mniej kolumn po kodowaniu.** Kodowanie *one-hot* zmiennej `cabin` dodałoby 187 kolumn, a zmiennej `CabinReduced` tylko 9. Mniejsza liczba cech zmniejsza ryzyko przeuczenia modelu i skraca czas treningu.
* **Liczniejsze kategorie.** W zmiennej `cabin` większość etykiet występuje tylko raz (57.53%), więc model nie ma podstaw do oszacowania ich wpływu. Po redukcji w większości kategorii jest wiele obserwacji dzielących ten sam pokład.
* **Łatwiejsza interpretacja.** Pokład jest czytelniejszą cechą niż pojedynczy numer kabiny i prawdopodobnie jest powiązany z klasą podróży oraz odległością od szalup ratunkowych.

### Potencjalne negatywne skutki redukcji

Rozkład nowej zmiennej, udział braków danych oraz zależność braków od klasy podróży pokazują poniższe komórki.

In [13]:
df["CabinReduced"].value_counts(dropna=False)

CabinReduced
NaN    1014
C        94
B        65
D        46
E        41
A        22
F        21
G         5
T         1
Name: count, dtype: int64

In [14]:
print('Odsetek braków danych w zmiennej cabin: {:.1%}'.format(df["cabin"].isnull().mean()))

Odsetek braków danych w zmiennej cabin: 77.5%


In [15]:
pd.crosstab(df["pclass"], df["cabin"].isnull(), rownames=["pclass"], colnames=["brak kabiny"])

brak kabiny,False,True
pclass,,
1.0,256,67
2.0,23,254
3.0,16,693


* **Utrata szczegółowości.** Po redukcji zostaje wyłącznie pokład, a numer kabiny (który mógł wskazywać położenie wzdłuż statku) znika. Sprawdziłem, że 41 z 295 wpisów zawiera kilka kabin, ale tylko 7 z nich obejmuje różne litery (F G63, F E57, F E46, F G73, F E69), i wszystkie zaczynają się od F. Prawdopodobnie druga litera oznacza sektor pokładu F, więc pierwsza litera poprawnie wskazuje pokład.
* **Bardzo liczne braki danych.** Braki dotyczą ponad 77% pasażerów, więc po redukcji jedna z 9 kategorii (brak danych) obejmuje większość zbioru. Nie jest to przypadek: kabiny odnotowano głównie dla pasażerów 1. klasy (tabela powyżej), więc sam brak wpisu może nieść informację o statusie pasażera. Warto go zatem zachować jako osobną kategorię, a nie usuwać.
* **Kategorie bardzo nieliczne.** Niektóre pokłady są reprezentowane przez pojedynczych pasażerów (np. pokład T przez jednego), co utrudnia oszacowanie ich wpływu na wynik.

In [16]:
df

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest,CabinReduced
0,1.0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0.0,0.0,24160,211.3375,B5,S,2,NaN,"St Louis, MO",B
1,1.0,1,"Allison, Master. Hudson Trevor",male,0.9167,1.0,2.0,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON",C
2,1.0,0,"Allison, Miss. Helen Loraine",female,2.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON",C
3,1.0,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON",C
4,1.0,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON",C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304,3.0,0,"Zabour, Miss. Hileni",female,14.5000,1.0,0.0,2665,14.4542,NaN,C,NaN,328.0,NaN,NaN
1305,3.0,0,"Zabour, Miss. Thamine",female,NaN,1.0,0.0,2665,14.4542,NaN,C,NaN,NaN,NaN,NaN
1306,3.0,0,"Zakarian, Mr. Mapriededer",male,26.5000,0.0,0.0,2656,7.2250,NaN,C,NaN,304.0,NaN,NaN
1307,3.0,0,"Zakarian, Mr. Ortin",male,27.0000,0.0,0.0,2670,7.2250,NaN,C,NaN,NaN,NaN,NaN


In [17]:
df.to_csv("data/titanic_cabinReduced.csv")